# 07d — Kết hợp

Notebook này ghép mọi thứ đã dựng thành một hệ thống, và cân nhắc xem ghép hết
có thật sự hơn ghép một phần hay không.

### Bốn chặng

```text
retrieve  ──►  dedup  ──►  rerank  ──►  fuse
  07a/05       07a           07b         day
```

### Gộp hai bảng xếp hạng như thế nào

Không cộng điểm của hai mô hình. Lý do nằm ở thang đo: `bm25` cộng trọng số
term nên điểm là số dương không chặn trên, còn mô hình ngữ nghĩa trả về cosine
trong khoảng −1 đến 1. Cộng hai số đó lại là so sánh hai thứ không cùng đơn vị.

Reciprocal Rank Fusion cộng **thứ hạng**, không cộng điểm:

$$\text{score}(d) = \sum_{\text{mỗi bảng}} \frac{1}{k + \text{rank}(d)}$$

Thứ hạng thì hai bảng nào cũng cùng thang đo. Tham số $k$ làm dịu phần đầu
bảng: với $k = 60$, khoảng cách giữa hạng 1 và hạng 2 đủ nhỏ để một tài liệu
muốn lên đầu phải được **nhiều bảng cùng ủng hộ**. Đó chính là mục đích của
việc gộp.

### Câu hỏi cần trả lời

| Câu hỏi | Cách trả lời |
| --- | --- |
| Gộp rồi xếp hạng lại có hơn chỉ xếp hạng lại không | So với hệ thống của `07b` |
| $k$ nên bằng bao nhiêu | Quét vài giá trị, chọn trên `fit` |
| Hệ thống nào đem nộp | Mục 5 |

### Nguyên tắc vẫn giữ

Thêm chặng thì thêm chỗ sai và thêm chi phí. Theo nguyên tắc số 5 của dự án,
hoà thì chọn cái đơn giản hơn. Một hệ thống bốn chặng chỉ được chọn khi nó hơn
hệ thống ba chặng một cách có ý nghĩa thống kê.

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
import reteco as R
import pipeline as P

DATA = Path(r"D:\RETECO-project\reteco_data\track1_tempo")
SYSTEMS = Path("..") / "systems"
GPU_JOBS = Path("..") / "gpu_jobs"
CACHE = Path("results")

P.configure(DATA, CACHE)
domains = P.domains()

# What earlier notebooks left behind.
have = {}
for name, path in (("dedup", CACHE / "dedup_summary.json"),
                   ("rerank", CACHE / "rerank_summary.json"),
                   ("dense", CACHE / "dense_summary.json")):
    have[name] = path.exists()
    print(f"{name:<10}{'ready' if have[name] else 'MISSING — run its notebook first'}")

assert have["dedup"], "notebook 07a has not been run"
dedup_summary = json.loads((CACHE / "dedup_summary.json").read_text(encoding="utf-8"))

In [ ]:
# --- Shared chart style -------------------------------------------------
# Same palette and helpers as every other notebook in the project, so a bar
# here means what a bar there means.
BLUE, ORANGE, TEAL, AMBER, PINK, VIOLET = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7")
SURFACE, INK, INK_SOFT, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_SOFT, "axes.labelsize": 9.5,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "font.size": 10,
    "figure.dpi": 120, "axes.linewidth": 0.9,
})


def finish(ax, title, subtitle=None, xlabel=None, ylabel=None,
           grid_axis="y", note=None):
    if subtitle:
        ax.set_title(subtitle, loc="left", pad=8, fontsize=9.5, color=INK_MUTED)
        ax.annotate(title, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, 24), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12.5,
                    fontweight="bold", color=INK, annotation_clip=False)
    else:
        ax.set_title(title, loc="left", pad=12, fontsize=12.5,
                     fontweight="bold", color=INK)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left" if grid_axis == "x" else "bottom"].set_color(AXIS)
    if grid_axis == "x":
        ax.spines["bottom"].set_visible(False)
    elif grid_axis:
        ax.spines["left"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    ax.tick_params(length=0)
    if note:
        ax.text(0, -0.34, note, transform=ax.transAxes, ha="left", va="top",
                fontsize=8.5, color=INK_MUTED)
    return ax


def label_bars(ax, bars, values, fmt="{:,.0f}", horizontal=True, pad=0.015):
    span = max(values) if len(values) else 1
    for bar, value in zip(bars, values):
        if horizontal:
            ax.text(bar.get_width() + span * pad,
                    bar.get_y() + bar.get_height() / 2, fmt.format(value),
                    va="center", ha="left", fontsize=8.5, color=INK_SOFT)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + span * pad, fmt.format(value),
                    ha="center", va="bottom", fontsize=8.5, color=INK_SOFT)


def legend_below(ax, ncol=2, y=-0.30):
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, y),
              ncol=ncol, fontsize=9.5, handlelength=1.1, handleheight=1.1,
              columnspacing=1.8)

---

## 1. Những gì đã có

Trước khi ghép, xem lại từng mảnh đã đo được gì. Bảng này đọc từ file điểm đã
lưu nên không chạy lại gì cả.

In [ ]:
# --- The parts, as measured ----------------------------------------------
P.table()
print()
P.metrics()

---

## 2. Gộp hai bảng xếp hạng

Chặng `fuse` nhận tên của những hệ thống khác đã chạy. Chúng phải có sẵn trong
`results/runs/`, nghĩa là mỗi cái đã là một hệ thống độc lập và mở ra xem riêng
được. Cách này giữ cho hệ thống gộp không thành một hộp đen.

Quét $k$ trên phần `fit`. $k$ nhỏ tin vào đầu bảng; $k$ lớn cần nhiều bảng cùng
ủng hộ hơn.

In [ ]:
# --- Sweep k ---------------------------------------------------------------
K_VALUES = [10, 30, 60, 120]
FUSE_DEPTH = 1000

stage1 = P.load_system(SYSTEMS / "03_stage1_deep.json")
stage1_digest = P.run(stage1, split="train")
P.score(stage1_digest)

sources = []
if have["dense"] and (SYSTEMS / "05_dense.json").exists():
    sources.append("dense")
    P.run(P.load_system(SYSTEMS / "05_dense.json"), split="train")

if not sources:
    print("No second ranking to fuse with. Run notebook 07c to produce one,")
    print("or fuse two sparse models instead by adding one here.")
    fuse_digests = {}
else:
    print(f"fusing {stage1['name']} with: {', '.join(sources)}")
    fuse_digests = {}
    for k in K_VALUES:
        system = dict(stage1, name=f"fuse_k{k}",
                      note=f"RRF of {stage1['name']} with {sources}, k={k}",
                      fuse={"kind": "rrf", "sources": sources, "k": k,
                            "depth": FUSE_DEPTH})
        (SYSTEMS / f"06_fuse_k{k}.json").write_text(
            json.dumps(system, indent=2), encoding="utf-8")
        digest = P.run(system, split="train")
        P.score(digest)
        fuse_digests[f"k={k}"] = digest

    print(f"\n{'k':<10}{'fit':>9}{'check':>9}{'all train':>11}"
          f"{'R@100':>9}{'ceiling':>10}")
    print("-" * 58)
    base = P.score(stage1_digest)
    print(f"{'khong gop':<10}{base['fit']['ndcg']:>9.4f}"
          f"{base['check']['ndcg']:>9.4f}{base['macro']['ndcg']:>11.4f}"
          f"{base['macro']['recall']:>9.4f}{base['macro']['ceiling']:>10.4f}")
    for name, digest in fuse_digests.items():
        r = P.score(digest)
        print(f"{name:<10}{r['fit']['ndcg']:>9.4f}{r['check']['ndcg']:>9.4f}"
              f"{r['macro']['ndcg']:>11.4f}{r['macro']['recall']:>9.4f}"
              f"{r['macro']['ceiling']:>10.4f}")
    print("-" * 58)

    best_k = max(fuse_digests, key=lambda n: P.score(fuse_digests[n])["fit"]["ndcg"])
    print(f"Highest on fit: {best_k}")
    if best_k in (f"k={K_VALUES[0]}", f"k={K_VALUES[-1]}"):
        print("That is the edge of the sweep. Widen K_VALUES before trusting it.")

In [ ]:
# --- Does fusing beat not fusing -----------------------------------------
if fuse_digests:
    print(f"{'comparison':<34}{'difference':>12}{'95% interval':>26}"
          f"{'verdict':>9}")
    print("-" * 81)
    fuse_significance = P.compare(fuse_digests[best_k], stage1_digest,
                                  part="fit")
    print("-" * 81)
    print()
    if fuse_significance["significant"] and fuse_significance["diff"] > 0:
        print("Fusing is ahead by more than noise. It earns its place.")
    else:
        print("Tie. By the project's rule, a tie picks the simpler system, so")
        print("the fusion stage does not go into the submission on this")
        print("evidence alone. It may still help AFTER reranking; section 3")
        print("tests that, which is a different question.")
else:
    fuse_significance = None

---

## 3. Thứ tự các chặng

Gộp rồi xếp hạng lại, hay xếp hạng lại rồi gộp? Hai thứ tự này làm hai việc
khác nhau:

| Thứ tự | Bộ xếp hạng lại nhìn thấy gì |
| --- | --- |
| Gộp trước | Danh sách ứng viên rộng hơn, gồm cả thứ ngữ nghĩa tìm ra |
| Xếp hạng lại trước | Chỉ những gì `bm25` tìm được |

Gộp trước là hợp lý hơn: bộ xếp hạng lại chỉ sắp xếp được những gì nó nhận,
nên cho nó nhiều lựa chọn hơn thì trần của nó cao hơn. Nhưng nó cũng đắt hơn,
vì số cặp phải chấm tăng lên.

Khung của dự án chạy các chặng theo thứ tự cố định `retrieve → dedup → rerank
→ fuse`, nên "gộp trước rồi xếp hạng lại" được dựng bằng cách lấy hệ thống gộp
làm tầng một của một gói việc GPU mới.

In [ ]:
# --- Export a GPU job over the fused list --------------------------------
FUSED_JOB = "rerank_fused_01"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if fuse_digests:
    fused_job = P.export_gpu_job(
        fuse_digests[best_k], GPU_JOBS / FUSED_JOB, FUSED_JOB,
        task="rerank", split="train", depth=1000,
        with_text=False, model=RERANK_MODEL)
    print()
    print("Run on the GPU machine:")
    print(f"    python gpu_worker.py {FUSED_JOB} "
          f"--data reteco_data/track1_tempo")
    print("then copy scores.jsonl back and run the next cell.")

In [ ]:
# --- Rerank the fused list ------------------------------------------------
fused_scores = GPU_JOBS / FUSED_JOB / "scores.jsonl"
full_digest = None

if fuse_digests and fused_scores.exists():
    P.import_gpu_scores(fused_scores, FUSED_JOB)
    k_value = int(best_k.split("=")[1])
    # fuse runs last in the stage order, so a system that fuses AND reranks
    # reranks first. To rerank the FUSED list, the fusion is the first stage's
    # own output, carried in by hash.
    full = {
        "name": "full_stack",
        "note": f"fuse(k={k_value}) then rerank with {RERANK_MODEL}",
        "retrieve": {"kind": "gpu_job", "job": FUSED_JOB, "depth": 1000},
    }
    (SYSTEMS / "07_full_stack.json").write_text(
        json.dumps(full, indent=2), encoding="utf-8")
    full_digest = P.run(full, split="train")
    P.score(full_digest)
    print("scored")
elif fuse_digests:
    print(f"{fused_scores} is not here yet; the cell above says how to make it.")

---

## 4. Bảng tổng: mọi hệ thống đã dựng

Đây là bảng dùng để chọn bài nộp. Sắp theo `fit`, vì `check` chỉ đọc sau khi
đã chọn.

In [ ]:
# --- Everything, in one table --------------------------------------------
records = P.table()
print()
P.metrics()
print()

complete = [r for r in records if r["n_topics"] == r["n_gold_queries"]]
if complete:
    leader = max(complete, key=lambda r: r["fit"]["ndcg"])
    print(f"Highest on fit, among systems answering every query:")
    print(f"  {leader['name']}  ({leader['hash']})")
    print(f"  fit {leader['fit']['ndcg']:.4f}   "
          f"check {leader['check']['ndcg']:.4f}   "
          f"all train {leader['macro']['ndcg']:.4f}")
    print()
    print("`check` was not used to pick this. It is read now, once, as a")
    print("sanity check: if it disagrees badly with `fit`, the choice was")
    print("fitted to the fitting half and will not hold up on dev.")
    gap = leader["fit"]["ndcg"] - leader["check"]["ndcg"]
    print(f"  fit minus check: {gap:+.4f}")

In [ ]:
# --- How each stage contributed ------------------------------------------
fuse_file = (SYSTEMS / f"06_fuse_{best_k.replace('=', '')}.json"
             if fuse_digests else None)

chain = []
for label, path in (
        ("1. bm25 tinh chinh", SYSTEMS / "01_bm25_tuned.json"),
        ("2. + lay sau, gop ban sao", SYSTEMS / "03_stage1_deep.json"),
        ("3. + xep hang lai", SYSTEMS / "04_rerank_cut1000.json"),
        ("4. + gop hai bang", fuse_file),
        ("5. + xep hang lai ban gop", SYSTEMS / "07_full_stack.json")):
    if path is None or not Path(path).exists():
        continue
    digest = P.run(P.load_system(path), split="train", verbose=False)
    chain.append((label, P.score(digest)["macro"]["ndcg"],
                  P.score(digest)["macro"]["ceiling"]))

if len(chain) > 1:
    fig, ax = plt.subplots(figsize=(9.6, 4.4))
    ys = np.arange(len(chain))
    values = [c[1] for c in chain]
    ceilings = [c[2] for c in chain]
    ax.barh(ys, ceilings, height=0.62, color=GRID, label="tran xep hang lai")
    bars = ax.barh(ys, values, height=0.62, color=BLUE, label="nDCG@10")
    ax.set_yticks(ys)
    ax.set_yticklabels([c[0] for c in chain], fontsize=9.5)
    ax.invert_yaxis()
    ax.set_xlim(0, max(ceilings) * 1.2)
    for y, (label, value, ceiling) in zip(ys, chain):
        step = "" if y == 0 else f"   ({value - chain[y - 1][1]:+.4f})"
        ax.text(ceiling * 1.02, y, f"{value:.4f}{step}", va="center",
                ha="left", fontsize=8.5, color=INK_SOFT)
    legend_below(ax, ncol=2, y=-0.18)
    finish(ax, f"Tu {chain[0][1]:.4f} len {chain[-1][1]:.4f} qua "
               f"{len(chain) - 1} buoc",
           subtitle="Phan xam la tran: khoang con lai cho buoc xep hang lai",
           xlabel="nDCG@10, trung binh theo nhom", grid_axis="x")
    plt.show()

---

## 5. Chọn hệ thống đem nộp

Ba điều kiện, theo đúng nguyên tắc của dự án:

1. Cao nhất trên phần `fit`
2. Trả lời đủ mọi câu truy vấn
3. Hơn hệ thống đơn giản hơn nó một cách có ý nghĩa thống kê

Điều kiện 3 là điều kiện lọc. Một hệ thống bốn chặng hơn hệ thống ba chặng
0,002 điểm thì không đáng thêm một chỗ để hỏng.

In [ ]:
# --- Apply the three rules -----------------------------------------------
ladder = [(name, digest) for name, digest in [
    ("bm25_tuned", P.run(P.load_system(SYSTEMS / "01_bm25_tuned.json"),
                         split="train", verbose=False)),
    ("stage1_deep", stage1_digest),
] if digest]
for extra, path in (("rerank", SYSTEMS / "04_rerank_cut1000.json"),
                    ("fuse", fuse_file),
                    ("full_stack", SYSTEMS / "07_full_stack.json")):
    if path and Path(path).exists():
        ladder.append((extra, P.run(P.load_system(path), split="train",
                                    verbose=False)))

print(f"{'system':<16}{'fit':>9}{'complete':>11}{'beats the one below':>22}")
print("-" * 58)
chosen = ladder[0]
for i, (name, digest) in enumerate(ladder):
    record = P.score(digest)
    complete_flag = record["n_topics"] == record["n_gold_queries"]
    verdict = "-"
    if i > 0:
        result = P.compare(digest, ladder[i - 1][1], part="fit", n_resamples=10_000)
        verdict = ("YES" if result["significant"] and result["diff"] > 0
                   else "tie")
        if verdict == "YES" and complete_flag:
            chosen = (name, digest)
    print(f"{name:<16}{record['fit']['ndcg']:>9.4f}"
          f"{('yes' if complete_flag else 'NO'):>11}{verdict:>22}")
print("-" * 58)
print()
print(f"Chosen for submission: {chosen[0]}  ({chosen[1]})")
print("It is the deepest system in the chain that is both complete and")
print("significantly ahead of the simpler system below it.")

In [ ]:
# --- Record the choice ----------------------------------------------------
hybrid_verdict = {
    "k_values": K_VALUES,
    "best_k": best_k if fuse_digests else None,
    "fuse_sources": sources if fuse_digests else [],
    "fuse_significance": fuse_significance,
    "chain": [{"label": c[0], "ndcg": c[1], "ceiling": c[2]} for c in chain],
    "ladder": [{"name": n, "hash": d,
                "fit": P.score(d)["fit"]["ndcg"],
                "ndcg": P.score(d)["macro"]["ndcg"]} for n, d in ladder],
    "chosen": {"name": chosen[0], "hash": chosen[1]},
}
(CACHE / "hybrid_summary.json").write_text(
    json.dumps(hybrid_verdict, indent=1), encoding="utf-8")
print(f"Saved {CACHE / 'hybrid_summary.json'}")
print()
print("Refresh the comparison page with:")
print("    python ../src/report.py --results results --open")

---

## 6. Kết luận

### Điều notebook này quyết định

Hệ thống nào đem nộp, và lý do. Lý do phải là một trong hai:

- Nó hơn hệ thống đơn giản hơn một cách có ý nghĩa thống kê
- Nó hoà, và nó chính là hệ thống đơn giản hơn

Không có lý do thứ ba. "Nó phức tạp hơn nên chắc tốt hơn" không phải lý do.

### Vì sao không ghép thêm nữa

Mỗi chặng thêm vào là thêm một chỗ có thể hỏng khi chạy trên `dev`, và là thêm
một tham số chọn trên `train`. Notebook `03b` đã mô phỏng được rằng chọn cái
tốt nhất trong 20 phương án ngang tài làm điểm của người thắng phồng lên
**+0,0280** — lớn hơn nhiều chênh lệch mà dự án này đang cân nhắc. Đó là lý do
mỗi bước đều phải qua kiểm định, và là lý do danh sách hệ thống nên dừng lại
khi các bước thêm vào không còn thắng rõ ràng.

### Bước tiếp theo

Notebook `08` chạy hệ thống đã chọn trên tập `dev` **đúng một lần**, xuất file
nộp, và tổng hợp toàn bộ số liệu của dự án.